# Testing the finetuned model

In [1]:
prompt = """Doctor: Hey, how are you doing today?

Patient: Hello doctor. I am feeling pain on the bottom right in my belly.

Doctor: How long has the pain been there?

Patient: It started yesterday evening and got worse during the night.

Doctor: Can you describe the pain? Is it sharp, dull, cramping, or something else?

Patient: It started as a dull ache, but now it feels sharp when I move or walk.

Doctor: On a scale from 1 to 10, how strong is the pain?

Patient: Around 7 out of 10.

Doctor: Have you noticed any nausea, vomiting, fever, or changes in appetite?

Patient: Yes, I feel nauseous and I did not want breakfast this morning. I also think I have a slight fever.

Doctor: Have you had diarrhea or constipation?

Patient: No diarrhea, but I have not gone to the bathroom since yesterday.

Doctor: Does anything make the pain better or worse?

Patient: Moving makes it worse. Lying still helps a little.

Doctor: Have you experienced this kind of pain before?

Patient: No, never this bad.

Doctor: Do you have any medical conditions or take any medications regularly?

Patient: No major medical conditions. I only take allergy medicine sometimes.

Doctor: Thank you. I would like to examine your abdomen now, especially the lower right side.

Patient: Okay.

Doctor: When I press here, does it hurt?

Patient: Yes, especially when you let go.

Doctor: I understand. Based on your symptoms and the examination, this could be appendicitis. I recommend blood tests and an abdominal scan as soon as possible.

Patient: Is it serious?

Doctor: It can become serious if untreated, but we caught it early. We will arrange further testing immediately.

Patient: Thank you, doctor.

Doctor: You're welcome. We will take good care of you."""

In [2]:
def build_messages(example: str, use_system_prompt: bool = True) -> list[dict]:
    messages = [
        {"role": "system",
        "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
        {"role": "user",   "content": example},
    ]
    if not use_system_prompt:
        messages = [messages[-1]]

    return messages

In [3]:
from vllm import LLM, SamplingParams
import os

In [4]:
SAMPLING = dict(temperature=0.0, max_tokens=1000)
sampling_params = SamplingParams(**SAMPLING)

In [6]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT") #modify
USER = os.getenv("SLURM_JOB_USER") #modify
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
input_model = "Qwen/Qwen3-4B-Instruct-2507"
model_output_name = f"{input_model}_finetuned"
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

In [8]:
llm = LLM(model=merged_output_dir, tensor_parallel_size=1, dtype="bfloat16")

INFO 06-05 11:49:22 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged'}
INFO 06-05 11:49:22 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 06-05 11:49:22 [model.py:1680] Using max model len 262144
INFO 06-05 11:49:22 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=89706) INFO 06-05 11:49:31 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged', speculative_config=None, tokenizer='/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, d

(EngineCore pid=89706) /opt/venv/lib/python3.12/site-packages/tensorizer/utils.py:23: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
(EngineCore pid=89706)   import pynvml


(EngineCore pid=89706) INFO 06-05 11:49:32 [parallel_state.py:1402] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.253.29.3:36679 backend=nccl
(EngineCore pid=89706) INFO 06-05 11:49:32 [parallel_state.py:1715] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=89706) INFO 06-05 11:49:33 [gpu_model_runner.py:4777] Starting to load model /scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged...
(EngineCore pid=89706) INFO 06-05 11:49:34 [rocm.py:538] Found incompatible backend(s) [TURBOQUANT] with AttentionType.DECODER. Overriding with ROCM_ATTN out of potential backends: ['ROCM_ATTN', 'TRITON_ATTN'].
(EngineCore pid=89706) WARNING 06-05 11:49:35 [compilation.py:1286] Op 'sparse_attn_indexer' not present in model, enabling with '+sparse_attn_indexer' has no effect


Loading pt checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading pt checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.75s/it]
Loading pt checkpoint shards: 100% Completed | 2/2 [00:04<00:00,  2.63s/it]
Loading pt checkpoint shards: 100% Completed | 2/2 [00:04<00:00,  2.50s/it]
(EngineCore pid=89706) 


(EngineCore pid=89706) INFO 06-05 11:49:40 [default_loader.py:384] Loading weights took 5.05 seconds
(EngineCore pid=89706) INFO 06-05 11:49:40 [gpu_model_runner.py:4879] Model loading took 7.67 GiB memory and 6.426481 seconds
(EngineCore pid=89706) INFO 06-05 11:49:49 [backends.py:1069] Using cache directory: /users/hintsala/.cache/vllm/torch_compile_cache/507b71bda5/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=89706) INFO 06-05 11:49:49 [backends.py:1128] Dynamo bytecode transform time: 8.34 s
(EngineCore pid=89706) INFO 06-05 11:50:00 [backends.py:376] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=89706) INFO 06-05 11:50:07 [backends.py:391] Compiling a graph for compile range (1, 8192) takes 17.92 s
(EngineCore pid=89706) INFO 06-05 11:50:09 [decorators.py:668] saved AOT compiled function to /users/hintsala/.cache/vllm/torch_compile_cache/torch_aot_compile/2490b123541c74084a45a65d2fb3061bce8e0eab0cd29e4ae714f3bec61481d7/rank_0_0/model
(Engin

...

(EngineCore pid=89706) WARNING 06-05 11:50:20 [chunked_prefill_paged_decode.py:400] Cannot use ROCm custom paged attention kernel, falling back to Triton implementation.


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 12.37it/s]


(EngineCore pid=89706) INFO 06-05 11:50:22 [gpu_model_runner.py:6133] Graph capturing finished in 7 secs, took 0.43 GiB
(EngineCore pid=89706) INFO 06-05 11:50:22 [core.py:299] init engine (profile, create kv cache, warmup model) took 42.34 s (compilation: 28.55 s)


(EngineCore pid=89706) The tokenizer you are loading from '/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore pid=89706) INFO 06-05 11:50:23 [vllm.py:840] Asynchronous scheduling is enabled.
(EngineCore pid=89706) INFO 06-05 11:50:23 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


## Inference with system prompt

In [9]:
outputs = llm.chat(build_messages(prompt), sampling_params, use_tqdm=True)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

The tokenizer you are loading from '/scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


INFO 06-05 11:50:45 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [10]:
print(outputs[0].outputs[0].text)

**REASON FOR VISIT:**  
The patient presented with acute right lower quadrant abdominal pain that began the previous evening and worsened during the night.

**PATIENT DETAILS AND HISTORY:**  
The patient is a 22‑year‑old female. She has no known chronic medical conditions and takes occasional allergy medication. She denies prior episodes of similar abdominal pain. No prior surgeries or significant lifestyle factors were reported.

**CURRENT STATUS:**  
She reports a dull‑to‑sharp right lower quadrant pain rated approximately 7/10, accompanied by nausea, anorexia, and a low-grade fever. She has not passed stool since the onset of symptoms and denies diarrhea. Physical examination reveals tenderness in the right lower quadrant, with rebound and guarding noted. Vital signs are not specified. Laboratory studies show a white blood cell count of 13,000/µL, a C‑reactive protein of 10.8 mg/dL, and an erythrocyte sedimentation rate of 30 mm/hr, all suggestive of inflammation. An abdominal ultra

## Inference without system prompt

In [11]:
outputs = llm.chat(build_messages(prompt, use_system_prompt=False), sampling_params, use_tqdm=True)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [12]:
print(outputs[0].outputs[0].text)

That’s a thorough and thoughtful clinical encounter. Here's a concise summary of the key points from the doctor-patient interaction, suitable for documentation or educational purposes:

---

**Chief Complaint:** Sudden onset of pain in the lower right abdomen, worsening over the past 24 hours.

**History of Present Illness:**  
The patient reports a dull ache that began yesterday evening and has become sharp with movement. Pain is rated 7/10. Associated symptoms include nausea, loss of appetite (refused breakfast), and a low-grade fever. There is no diarrhea, but bowel movements have been absent since yesterday. Pain is exacerbated by walking and slightly relieved by lying still. The patient denies prior episodes of similar pain.

**Past Medical History:** No significant chronic medical conditions; occasional use of allergy medication.

**Physical Examination:**  
Abdominal exam reveals tenderness, especially in the right lower quadrant, with rebound tenderness on release of pressure—c